# Module 3 : Text mining, les concepts

*Mercredi 24 juin 2026, 9h-12h · Intervenant : Corentin Vande Kerckhove*

**CONSEIL :** Sauvegarde une copie de ce notebook dans ton Drive avant de commencer !

**Objectifs d'apprentissage**
- **Collecter** un corpus de texte en ligne par scraping (`requests` + `BeautifulSoup`)
- **Partie 1, comprendre** chaque étape sur **3 annonces** : tokenization, bag of words, réduction du vocabulaire, puis **trois façons de vectoriser** (TF-IDF, Word2Vec, SentenceTransformer/BERT) comparées par **similarité cosinus**
- **Partie 2, passer à l'échelle** : appliquer la **meilleure** méthode aux **1 200 annonces** pour tester des hypothèses de recherche

**Pré-requis :** J2 matin (feature, vectorisation, token, embedding).

**Paquets requis :** `requests`, `beautifulsoup4`, `scikit-learn`, `nltk`, `gensim`, `sentence-transformers`, `wordcloud`, `matplotlib`, `pandas`

> 🍷 **Clin d'œil au cours.** On a vu qu'on pouvait recommander un **vin** pour accompagner un **plat** en comparant leurs descriptions *par le sens*. Le moteur derrière (transformer du texte en vecteurs, puis mesurer leur proximité) est exactement celui de ce notebook. On l'applique ici aux descriptions d'annonces Airbnb, mais le mécanisme est le même.

## La problématique : quels imaginaires du voyage sur Airbnb ?

Les plateformes comme Airbnb ne vendent pas seulement un hébergement, mais aussi une certaine **vision du voyage**. Certaines annonces insistent sur le confort, les équipements, le luxe du logement ; d'autres mettent en avant l'authenticité, la découverte du quartier, l'expérience locale.

On va analyser les **descriptions d'annonces** de trois villes (Bruxelles, Paris, Amsterdam, données [Inside Airbnb](https://insideairbnb.com)) pour comprendre quels discours les hôtes mobilisent. Fil rouge : transformer ces descriptions en vecteurs, construire des **requêtes thématiques** et mesurer leur proximité avec chaque annonce pour tester trois hypothèses :

- **H1** : les logements les plus **chers** sont davantage associés à un discours de **confort / luxe** ;
- **H2** : les logements en quartier **central** sont davantage associés à un discours d'**expérience locale** ;
- **H3** : les **chambres privées** mettent davantage en avant l'**hospitalité** et la rencontre.

On commence petit (3 annonces) pour **comprendre**, puis on passe à tout le dataset pour **conclure**.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/ressources/j3/1_matin/img/m3/m3-opener.jpg" width="440" alt="Valise ouverte">

*Running example : les imaginaires du voyage dans les annonces Airbnb. Photo : valise ouverte, emmamccleary, CC BY 2.0.*

## D'abord, le tableau d'ensemble

Avant de mettre les mains dans le code, une **démo (5 min)** pour voir où on va. L'explication interactive du *Financial Times*, [« Generative AI exists because of the transformer »](https://ig.ft.com/generative-ai/), déroule en images toute la chaîne qu'on va construire à la main dans ce module : un texte est découpé en **tokens**, chaque mot devient un **vecteur**, et des mots proches par le sens se retrouvent proches dans l'espace (avec la fameuse analogie `roi - homme + femme ≈ reine`, qu'on recroisera avec Word2Vec).

C'est exactement le passage « du texte aux nombres » qu'on va outiller ici, étape par étape.

## Setup

In [ ]:
# Sur Colab : exécute cette cellule pour installer les paquets (déjà installés sur un poste local)
!pip install -q requests beautifulsoup4 scikit-learn nltk gensim sentence-transformers wordcloud matplotlib pandas

import os
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

## Collecter : aller chercher les données en ligne (scraping)

Première étape de tout projet de text mining : **récupérer un corpus**. Les données vivent sur [Inside Airbnb](https://insideairbnb.com/get-the-data/), qui publie pour chaque ville un fichier `listings.csv.gz`.

Le scraping se fait en deux temps :
1. `requests` télécharge le **HTML** d'une page web ;
2. `BeautifulSoup` **parse** ce HTML pour en extraire ce qui nous intéresse (ici, les liens de téléchargement).

> **Bonnes pratiques** : on lit les conditions d'utilisation et le `robots.txt`, on s'identifie via un `User-Agent`, et on espace les requêtes pour ne pas surcharger le serveur.

In [ ]:
DATA_PAGE = "https://insideairbnb.com/get-the-data/"
headers = {"User-Agent": "Mozilla/5.0 (cours text mining)"}

html = requests.get(DATA_PAGE, headers=headers, timeout=60).text
soup = BeautifulSoup(html, "html.parser")

# On parcourt toutes les balises <a> et on garde les liens vers un listings.csv.gz
liens = [a["href"] for a in soup.find_all("a", href=True)
         if a["href"].endswith("listings.csv.gz")]

print(f"{len(liens)} liens de villes trouvés sur la page. Exemples :")
for lien in liens[:4]:
    print(" -", lien)

On télécharge ensuite le fichier avec `requests` et on le lit avec pandas. Ces fichiers sont **lourds** (Paris ≈ 80 000 annonces), alors on construit un **échantillon léger** : 400 annonces par ville, descriptions **en anglais** (les hôtes visent une clientèle internationale), prix renseigné.

La fonction ci-dessous fait tout, **directement depuis Inside Airbnb** : aucun fichier à uploader. Pour aller plus vite, on essaie d'abord une copie déjà hébergée (GitHub), puis une copie locale, et seulement en dernier recours on reconstruit le corpus depuis la source.

In [ ]:
import io, html as html_lib

EN = {"the","and","with","you","your","for","this","is","are","of","to","in","near","from","we","our"}
FR = {"le","la","les","et","avec","vous","votre","pour","une","un","des","du","est","dans","près","nous"}

def is_english(texte):
    mots = re.findall(r"[a-zà-ÿ]+", texte.lower())
    en = sum(m in EN for m in mots); fr = sum(m in FR for m in mots)
    return en >= 3 and en > fr * 1.5

def clean_text(texte):
    texte = re.sub(r"<[^>]+>", " ", html_lib.unescape(str(texte)))
    return re.sub(r"\s+", " ", texte).strip()

def build_corpus(n_par_ville=400):
    villes = {"Brussels": "/brussels/", "Paris": "/paris/", "Amsterdam": "/amsterdam/"}
    # Le dernier dump Paris a des prix vides : on épingle une date connue-bonne.
    epingles = {"Paris": "https://data.insideairbnb.com/france/ile-de-france/paris/2025-06-06/data/listings.csv.gz"}
    cols = ["id","description","neighbourhood_cleansed","room_type","price","latitude","longitude"]
    frames = []
    for ville, jeton in villes.items():
        url = epingles.get(ville) or next(l for l in liens if jeton in l)
        brut = pd.read_csv(io.BytesIO(requests.get(url, headers=headers, timeout=180).content),
                           compression="gzip")[cols].dropna(subset=["description"])
        brut["description"] = brut["description"].map(clean_text)
        brut = brut[brut["description"].str.split().str.len() >= 15]
        brut = brut[brut["description"].map(is_english)]
        brut["price"] = pd.to_numeric(brut["price"].astype(str).str.replace(r"[^0-9.]", "", regex=True),
                                      errors="coerce")
        brut = brut.dropna(subset=["price"])
        brut["city"] = ville
        frames.append(brut.sample(min(n_par_ville, len(brut)), random_state=42))
    return (pd.concat(frames, ignore_index=True)
              .rename(columns={"neighbourhood_cleansed": "neighbourhood"}))

# Sur Colab, le chemin local n'existe pas : on essaie d'abord la copie hébergée (GitHub),
# puis la copie locale, et en dernier recours on reconstruit depuis Inside Airbnb.
URL = "https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/data/airbnb/airbnb_listings.csv"
LOCAL = "../../../data/airbnb/airbnb_listings.csv"
try:
    annonces = pd.read_csv(URL)
    print("Données chargées depuis : GitHub (copie hébergée)")
except Exception:
    try:
        annonces = pd.read_csv(LOCAL)
        print("Données chargées depuis : copie locale")
    except Exception:
        print("Construction du corpus directement depuis Inside Airbnb...")
        annonces = build_corpus()
        print("Données chargées depuis : Inside Airbnb (reconstruction)")

print(f"{len(annonces)} annonces : {annonces['city'].value_counts().to_dict()}")
annonces[["city", "neighbourhood", "room_type", "price", "description"]].head(3)

# Partie 1 : Tout comprendre sur 3 annonces

Avant de lâcher les algorithmes sur 1 200 annonces, on prend **trois descriptions** et on suit chaque étape à la main. On a choisi deux annonces qui *semblent* proches (deux appartements parisiens haut de gamme) et une qui *semble* différente (un logement bruxellois « tout simple »). Tout le jeu : **vérifier cette intuition avec des chiffres**.

In [ ]:
trois = pd.DataFrame({
    "annonce": ["A : Paris, design", "B : Paris, design", "C : Bruxelles, simple"],
    "description": [
        "Discover the best of Paris, with this two bedroom apartment in Auteuil. "
        "It will be easy to simply show up and start living in this elegantly Blueground "
        "furnished apartment with its fully-equipped kitchen, beautiful living room, and "
        "our dedicated on-the-ground support.",
        "Show up and start living from day one in Paris with this cozy one bedroom "
        "Blueground apartment. You will love coming home to this thoughtfully furnished, "
        "beautifully designed, and fully-equipped Le Marais home.",
        "Keep it simple at this peaceful and centrally-located place. Located on the first "
        "floor with a terrace. Sleeps 3 people (1 double bed and 1 sofa bed). Tourist taxes "
        "included.",
    ],
})
for _, r in trois.iterrows():
    print(f"### {r.annonce}\n{r.description}\n")

## Tokenization

Un ordinateur ne « lit » pas une phrase : il manipule des **tokens** (des mots). La première opération **découpe** chaque description : tout en minuscules, on ne garde que les suites de lettres, on jette les tokens trop courts.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/ressources/j3/1_matin/img/m3/m3-tokenization.png" width="560" alt="Schéma tokenization">

*Tokenization : le texte brut est découpé en une liste de tokens.*

In [ ]:
def tokenize(texte):
    mots = re.findall(r"[a-z]+", texte.lower())     # minuscules + suites de lettres
    return [m for m in mots if len(m) > 2]          # on jette les tokens trop courts

for _, r in trois.iterrows():
    print(f"{r.annonce:>22} -> {tokenize(r.description)[:12]}")

## Bag of words → la matrice term-document

On oublie l'ordre des mots et on compte **combien de fois** chaque mot apparaît dans chaque annonce. Empilées, ces lignes forment la **term-document matrix** (TDM) : une ligne par annonce, une colonne par mot.

Avec seulement 3 annonces, on peut **regarder la matrice**. Affichons les mots présents dans **au moins deux** annonces : c'est là que se joue la ressemblance.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/ressources/j3/1_matin/img/m3/m3-bow.png" width="560" alt="Schéma bag of words">

*Bag of words : chaque annonce devient une ligne de comptages de mots.*

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(tokenizer=tokenize, token_pattern=None)
M = cv.fit_transform(trois["description"])
tdm = pd.DataFrame(M.toarray(), index=trois["annonce"],
                   columns=cv.get_feature_names_out())

print("Matrice complète :", tdm.shape, "(3 annonces × mots du vocabulaire)\n")
partages = tdm.loc[:, (tdm > 0).sum() >= 2]    # mots présents dans >= 2 annonces
print("Mots partagés par au moins 2 annonces :")
partages

A et B partagent déjà un vocabulaire de luxe (*living*, *furnished*, *equipped*, *blueground*…) ; C partage surtout des mots passe-partout.

## Réduire le vocabulaire

La matrice a beaucoup de colonnes inutiles. On réduit le vocabulaire (= le nombre de **dimensions**) avec **trois leviers** :

1. **retirer les stopwords** : mots outils très fréquents et peu informatifs (*the*, *and*, *with*…) ;
2. **lemmatiser** : ramener chaque mot à sa forme de base (`apartments` → `apartment`) pour **regrouper les variantes** ;
3. **filtrer par fréquence** : ne garder que les mots présents dans assez d'annonces.

La lemmatisation utilise WordNet (via `nltk`) :

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
lemmatiseur = WordNetLemmatizer()

for mot in ["apartments", "studios", "rooms", "houses", "amenities", "terraces"]:
    print(f"{mot:>12} -> {lemmatiseur.lemmatize(mot)}")

In [ ]:
def prepare(texte, retirer_stop=True, lemmatiser=True):
    mots = tokenize(texte)
    if retirer_stop:
        mots = [m for m in mots if m not in ENGLISH_STOP_WORDS]
    if lemmatiser:
        mots = [lemmatiseur.lemmatize(m) for m in mots]
    return mots

def vocab_size(textes, min_df=1, **kw):
    cv = CountVectorizer(tokenizer=lambda t: prepare(t, **kw),
                         token_pattern=None, min_df=min_df)
    return cv.fit_transform(textes).shape[1]

t = trois["description"]
etapes = {
    "tokens bruts": vocab_size(t, retirer_stop=False, lemmatiser=False),
    "− stopwords": vocab_size(t, retirer_stop=True, lemmatiser=False),
    "− stopwords + lemmatisation": vocab_size(t, retirer_stop=True, lemmatiser=True),
    "+ filtre fréquence (présent dans ≥ 2 annonces)": vocab_size(t, min_df=2),
}
pd.Series(etapes, name="taille du vocabulaire (dimensions)").to_frame()

## Vectoriser : trois façons de transformer le texte en nombres

Pour **comparer** des annonces, il faut les représenter par des **vecteurs**. Trois familles de méthodes, de la plus simple à la plus riche :

**1. TF-IDF** : un vecteur **creux** (une dimension par mot du vocabulaire, pondérée par la rareté du mot). Deux textes sont proches s'ils **partagent des mots**. Transparent, mais **aveugle aux synonymes** (*flat* et *apartment* sont deux dimensions sans rapport).

**2. Word2Vec** : un vecteur **dense par mot**, appris sur un grand corpus (des mots qui apparaissent dans des contextes similaires ont des vecteurs proches). Pour représenter une annonce, on **moyenne** les vecteurs de ses mots. Capture le sens des mots, mais ignore l'ordre et le contexte (un mot a toujours le même vecteur).

**3. SentenceTransformer (SBERT)** : un modèle de la famille **BERT**. BERT est un grand réseau pré-entraîné qui lit une phrase **en contexte** (le vecteur d'un mot dépend de ses voisins). Brut, BERT renvoie un vecteur par *token*, mal adapté pour comparer des phrases. **SBERT** est un BERT **affiné** (*fine-tuné*) pour que deux phrases de sens proche aient des vecteurs proches, exactement notre besoin. La librairie `sentence-transformers` charge un tel modèle (`all-MiniLM-L6-v2`).

| Méthode | Vecteur | Capture | Limite principale |
|---|---|---|---|
| TF-IDF | creux, par mot | chevauchement de mots | aveugle aux synonymes |
| Word2Vec | dense, moyenne des mots | sens des mots | ignore le contexte / l'ordre |
| SBERT (BERT fine-tuné) | dense, par phrase | sens de la **phrase entière** | modèle lourd (pré-entraîné) |

On va donner à chaque méthode les **3 annonces** et regarder la **matrice de similarité cosinus** (1 = très proche, 0 = sans rapport).

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/ressources/j3/1_matin/img/m3/m3-embeddings.png" width="560" alt="Schéma espace d'embeddings">

*Embeddings : les mots proches par le sens se regroupent dans l'espace vectoriel.*

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def cosine_matrix(vecteurs):
    return pd.DataFrame(cosine_similarity(vecteurs).round(2),
                        index=trois["annonce"], columns=trois["annonce"])

### Méthode 1 : TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(tokenizer=prepare, token_pattern=None)
T = tf.fit_transform(trois["description"])
vocab = np.array(tf.get_feature_names_out())

# au passage, le mot le plus distinctif de chaque annonce
for i, nom in enumerate(trois["annonce"]):
    top = T[i].toarray().ravel().argsort()[::-1][:5]
    print(f"{nom:>22} -> {', '.join(vocab[top])}")

cosine_matrix(T)

### Méthode 2 : Word2Vec (moyenne des vecteurs de mots)

In [ ]:
from gensim.models import Word2Vec

# Word2Vec a normalement besoin d'un corpus très volumineux (des millions de mots).
# Ici on l'entraîne sur seulement 1 200 descriptions courtes : les voisins appris sont
# donc illustratifs et bruités, pas de qualité production. C'est précisément ce qui
# motive le passage à un modèle pré-entraîné (SBERT) juste après.
w2v = Word2Vec([prepare(t) for t in annonces["description"]],
               vector_size=80, window=5, min_count=5, workers=2, epochs=40, seed=42)
print("Exemple, proches de 'metro' :",
      ", ".join(w for w, _ in w2v.wv.most_similar("metro", topn=5)), "\n")

def w2v_vector(texte):
    vs = [w2v.wv[m] for m in prepare(texte) if m in w2v.wv]
    return np.mean(vs, axis=0) if vs else np.zeros(w2v.vector_size)

W = np.vstack([w2v_vector(d) for d in trois["description"]])
cosine_matrix(W)

### Méthode 3 : SentenceTransformer (SBERT)

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")
emb3 = model.encode(trois["description"].tolist())
cosine_matrix(emb3)

Les trois méthodes s'accordent : **A et B sont les plus proches**, C est à part. Jusqu'ici, n'importe laquelle ferait l'affaire.

### Laquelle garder pour tester les hypothèses ?

En partie 2, on ne comparera pas les annonces entre elles, mais chaque annonce à une **requête** écrite librement (« an upscale designer apartment… »), avec **d'autres mots** que les annonces. C'est là que les méthodes se départagent :

In [ ]:
requete = "an upscale designer apartment with high-end furnishings"

sim_tfidf = cosine_similarity(tf.transform([requete]), T).ravel()
sim_sbert = util.cos_sim(model.encode([requete]), emb3).numpy().ravel()

pd.DataFrame({"TF-IDF": sim_tfidf.round(2), "SBERT": sim_sbert.round(2)},
             index=trois["annonce"])

**TF-IDF juge B presque hors-sujet** : B emploie d'autres mots que la requête (« thoughtfully furnished, beautifully designed ») et ne partage donc presque rien avec elle. **SBERT** comprend que A *et* B décrivent toutes deux un logement design haut de gamme, et les classe au coude-à-coude devant C.

Pour faire correspondre des annonces à des requêtes thématiques, **on garde SBERT** : c'est la seule des trois qui compare par le **sens** plutôt que par les mots exacts.

### Hack Time

- Change la `requete` (par ex. `"a cheap simple room"`) : quelle annonce remonte ? TF-IDF et SBERT sont-ils d'accord ?
- Ajoute une 4ᵉ annonce au DataFrame `trois` et regarde de laquelle elle se rapproche (SBERT).

**Défi optionnel** : trouve une requête où TF-IDF et SBERT classent les annonces dans un ordre différent, et explique pourquoi.

In [ ]:
# Votre code ici

# Partie 2 : Vérifier les hypothèses sur tout le dataset

On réutilise les fonctions de la partie 1 sur les **1 200 annonces**. Pour H1, H2, H3, on garde **SBERT**, la méthode la plus fidèle au sens (cf. partie 1).

## Mots distinctifs par ville (TF-IDF à l'échelle)

Vérification rapide que tout passe à l'échelle : les mots au TF-IDF moyen le plus élevé, ville par ville (même `prepare`, juste plus de données).

In [ ]:
tfidf = TfidfVectorizer(tokenizer=prepare, token_pattern=None, min_df=5, max_df=0.5)
X_tfidf = tfidf.fit_transform(annonces["description"])
vocab_full = np.array(tfidf.get_feature_names_out())
print("Vocabulaire après réduction :", len(vocab_full), "mots\n")

def distinctive_words(masque, k=10):
    moyenne = np.asarray(X_tfidf[masque].mean(axis=0)).ravel()
    return ", ".join(vocab_full[moyenne.argsort()[::-1][:k]])

for ville in ["Brussels", "Paris", "Amsterdam"]:
    print(f"{ville:>10} : {distinctive_words((annonces['city'] == ville).values)}")

### Hack Time

- Calcule les mots distinctifs par **type de logement** (`room_type`) plutôt que par ville.

**Défi optionnel** : compare les mots distinctifs d'une même ville selon la tranche de prix et vois si le vocabulaire « de luxe » apparaît.

In [ ]:
# Votre code ici
# Indice : distinctive_words((annonces['room_type'] == 'Private room').values)

## H1, H2, H3 avec SBERT

On encode chaque annonce avec SBERT, puis on la compare à des **requêtes thématiques** qui incarnent nos hypothèses. On croise ensuite les scores avec le prix, la centralité et le type de logement.

In [ ]:
emb_annonces = model.encode(annonces["description"].tolist(),
                            show_progress_bar=False, batch_size=64)

requetes = {
    "confort_luxe": "a luxurious, comfortable and well-equipped home",
    "experience_locale": "an authentic local experience to explore the neighbourhood",
    "hospitalite": "a welcoming host, hospitality and meeting local people",
}
emb_requetes = model.encode(list(requetes.values()))

sims = util.cos_sim(emb_annonces, emb_requetes).numpy()
for i, nom in enumerate(requetes):
    annonces[f"sim_{nom}"] = sims[:, i]

annonces[["city", "price", "room_type",
          "sim_confort_luxe", "sim_experience_locale", "sim_hospitalite"]].head()

### H1 : Prix élevé ↔ discours de confort / luxe

In [ ]:
annonces["tranche_prix"] = pd.qcut(annonces["price"], 4,
                                   labels=["bas", "moyen", "élevé", "très élevé"],
                                   duplicates="drop")
grp = annonces.groupby("tranche_prix", observed=True)["sim_confort_luxe"]
h1, h1_std = grp.mean(), grp.std()
print(pd.DataFrame({"moyenne": h1.round(3), "écart-type": h1_std.round(3)}))
h1.plot(kind="bar", rot=0, yerr=h1_std, capsize=4, ylabel="similarité moyenne",
        title="H1 : discours confort/luxe selon le prix"); plt.show()
print("Attention : ne surinterprète pas un petit écart entre groupes sur cet échantillon "
      "(les barres d'erreur se chevauchent souvent).")

### H2 : Quartier central ↔ discours d'expérience locale

On mesure la distance de chaque logement au centre de sa ville, puis on compare les annonces **centrales** (plus proches que la médiane de leur ville) aux **périphériques**.

In [ ]:
centres = {"Brussels": (50.8467, 4.3525),
           "Paris": (48.8566, 2.3522),
           "Amsterdam": (52.3676, 4.9041)}

def distance_to_center(row):
    clat, clon = centres[row["city"]]
    # Distance à vol d'oiseau en degrés de lat/long, PAS une vraie distance routière en km.
    # Suffisant ici pour classer les annonces de la plus centrale à la plus périphérique.
    return np.hypot(row["latitude"] - clat, row["longitude"] - clon)

annonces["dist_centre"] = annonces.apply(distance_to_center, axis=1)
annonces["central"] = annonces.groupby("city")["dist_centre"].transform(
    lambda d: d < d.median())

grp2 = annonces.groupby("central")["sim_experience_locale"]
h2, h2_std = grp2.mean(), grp2.std()
h2.index = h2_std.index = ["périphérique", "central"]
print(pd.DataFrame({"moyenne": h2.round(3), "écart-type": h2_std.round(3)}))
h2.plot(kind="bar", rot=0, yerr=h2_std, capsize=4, ylabel="similarité moyenne",
        title="H2 : discours d'expérience locale selon la centralité"); plt.show()
print("Attention : ne surinterprète pas un petit écart entre groupes sur cet échantillon "
      "(les barres d'erreur se chevauchent souvent).")

### H3 : Chambre privée ↔ discours d'hospitalité

In [ ]:
sous = annonces[annonces["room_type"].isin(["Entire home/apt", "Private room"])]
grp3 = sous.groupby("room_type")["sim_hospitalite"]
h3, h3_std = grp3.mean(), grp3.std()
print(pd.DataFrame({"moyenne": h3.round(3), "écart-type": h3_std.round(3)}))
h3.plot(kind="bar", rot=0, yerr=h3_std, capsize=4, ylabel="similarité moyenne",
        title="H3 : discours d'hospitalité selon le type de logement"); plt.show()
print("Attention : ne surinterprète pas un petit écart entre groupes sur cet échantillon "
      "(les barres d'erreur se chevauchent souvent).")

### Hack Time

- Invente une **nouvelle requête thématique** (par ex. `"a quiet place to work remotely"`), encode-la et calcule sa similarité avec chaque annonce.
- Quelle ville obtient les scores les plus élevés ?

**Défi optionnel** : reprends une de tes requêtes et trace sa similarité moyenne par ville avec des barres d'erreur, comme pour H1/H2/H3.

In [ ]:
# Votre code ici

### Interpréter, avec prudence

Sur notre échantillon, **H3** ressort nettement (les chambres privées parlent plus d'hospitalité), **H1** est faiblement soutenue, **H2** ne l'est pas. Les écarts sont parfois ténus : petit corpus, requêtes volontairement génériques, un seul modèle d'embedding. C'est tout l'intérêt de la démarche (affiner les requêtes, agrandir le corpus, croiser les modèles) plutôt que de conclure trop vite. Le text mining **outille** l'analyse, il ne remplace pas l'interprétation du chercheur.

## Récap module 3

| Méthode | Représentation | Capture le sens ? | Ce qu'on en fait |
|---|---|---|---|
| Bag of words / TDM | creuse (comptages) | non | compter, repérer le vocabulaire |
| Réduction (stopwords, lemmatisation, fréquence) | creuse, compacte | non | vocabulaire propre et léger |
| TF-IDF | creuse (pondérée) | non | mots **distinctifs**, similarité par mots |
| Word2Vec | dense (par mot) | un peu | proximité de mots |
| SBERT (BERT fine-tuné) | dense (par phrase) | oui | **similarité** annonce ↔ requête, test d'hypothèses |

On a comparé les méthodes **sur 3 annonces**, choisi SBERT, puis tout appliqué **aux 1 200** pour répondre à une vraie question de recherche.

→ Direction le **notebook 2** : cinq **applications** concrètes du text mining (nuage de mots, sentiment, classification, recherche sémantique, biais), sur un corpus de critiques de films.